In [1]:
# Existing Kaggle dataset input path (no download or KaggleHub call).
from pathlib import Path
DATASET_ROOT = Path("/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset")
if not DATASET_ROOT.is_dir():
    DATASET_ROOT = Path("/kaggle/input/garbage-classification-image-dataset")
print("Dataset:", DATASET_ROOT)


Dataset: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset


In [2]:
# ==========================================================
# ResNet-50 Stage 2: torchvision model structure, frozen Stage 1 split; performance only
# No encoded manifest or additional ZIP input.
# ==========================================================
import os, sys, subprocess, importlib.util, random, copy, hashlib, json, zipfile, platform
from pathlib import Path
from collections import defaultdict

# Set GPU visibility before importing torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for module, package in [("docx", "python-docx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from docx import Document

SEEDS = [7, 49, 112, 345, 476]
BATCH_SIZE, MAX_EPOCHS, PATIENCE, MIN_DELTA = 32, 10, 3, 1e-4
OUT = Path("/kaggle/working/ResNet50_Stage2_Performance")
# Create the parent directory FIRST (fixes FileNotFoundError).
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "Model_Checkpoints").mkdir(parents=True, exist_ok=True)

# ----------------------------
# Dataset: same Kaggle path as uploaded notebook
# ----------------------------
# DATASET_ROOT is set to the attached Kaggle folder in the first cell.
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError("Kaggle input folder is unavailable; attach the original dataset in Kaggle Add Input.")
search_roots = [DATASET_ROOT]
for archive in sorted(DATASET_ROOT.rglob("*.zip")):
    target = Path("/kaggle/working/remnet_source") / archive.stem
    if not (target / ".extract_complete").exists():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                if not (target / entry.filename).resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe ZIP entry: {entry.filename}")
            z.extractall(target)
        (target / ".extract_complete").touch()
    search_roots.append(target)

split_dirs = []
for root in search_roots:
    for parent in [root, *sorted((p for p in root.rglob("*") if p.is_dir()), key=str)]:
        children = {p.name.lower(): p for p in parent.iterdir() if p.is_dir()}
        tr, te = children.get("training") or children.get("train"), children.get("testing") or children.get("test")
        if tr and te and {p.name for p in tr.iterdir() if p.is_dir()} == {p.name for p in te.iterdir() if p.is_dir()}:
            split_dirs.append((tr, te))
if not split_dirs:
    raise FileNotFoundError("Training/Testing class folders were not found in the original Kaggle dataset.")
TRAIN_DIR, TEST_DIR = split_dirs[0]
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
assert len(CLASS_NAMES) == 6, f"Expected 6 classes, got {CLASS_NAMES}"
print("Source Training:", TRAIN_DIR, "\nSource Testing:", TEST_DIR)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stage 1 cleaning: one representative per identical-image group;
# quarantine groups with conflicting labels, and remove the four
# previously visually confirmed near-duplicates.
# The four exclusions are plain filenames, not an encoded manifest.
visual_exclusions = {
    "Training/metal/metal394.jpg", "Training/paper/paper345.jpg",
    "Training/paper/paper390.jpg", "Training/plastic/plastic137.jpg"
}
by_hash = defaultdict(list)
for partition, root in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    for cls in CLASS_NAMES:
        for path in sorted((root / cls).iterdir()):
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                by_hash[file_sha256(path)].append({"path": path, "class": cls,
                                      "source": f"{partition}/{cls}/{path.name}",
                                      "original_partition": partition})

cleaned = []
conflicts = 0
for digest, group in by_hash.items():
    if len({item["class"] for item in group}) != 1:
        conflicts += len(group)     # Exclude conflicting-label images entirely.
        continue
    item = min(group, key=lambda x: x["source"])  # Prefer Testing over Training.
    cleaned.append({**item, "sha256": digest})

# The Stage 1 80/20 stratified train/validation split was formed BEFORE
# removing 4 visually confirmed near-duplicates. Sort SHA-256 first to
# reproduce exactly the Stage 1 split without embedding any manifest.
original_test = sorted((x for x in cleaned if x["original_partition"] == "Testing"),
                       key=lambda x: x["source"])
train_pool = sorted((x for x in cleaned if x["original_partition"] == "Training"),
                    key=lambda x: x["sha256"])
if (len(cleaned), len(train_pool), len(original_test), conflicts) != (2910, 2447, 463, 6):
    raise ValueError("Dataset differs from audited Stage 1 source; check Kaggle dataset version. "
                     f"Pre-visual counts: total={len(cleaned)}, pool={len(train_pool)}, "
                     f"test={len(original_test)}, conflicting images={conflicts}.")

training, validation = train_test_split(
    train_pool, test_size=0.2, random_state=42,
    stratify=[x["class"] for x in train_pool])
training = sorted((x for x in training if x["source"] not in visual_exclusions),
                  key=lambda x: x["source"])
validation = sorted((x for x in validation if x["source"] not in visual_exclusions),
                    key=lambda x: x["source"])
assert (len(training), len(validation), len(original_test)) == (1956, 487, 463)
assert len({x["sha256"] for x in training + validation + original_test}) == 2906

split_rows = []
for name, group in [("Training", training), ("Validation", validation), ("Testing", original_test)]:
    for item in group:
        split_rows.append({"split": name, "source_path": item["source"],
                           "class": item["class"], "sha256": item["sha256"]})
split_df = pd.DataFrame(split_rows).sort_values(["split", "source_path"])
split_df.to_csv(OUT / "REMNET_Split_Used.csv", index=False)
split_sha = hashlib.sha256(split_df.to_csv(index=False).encode()).hexdigest()
print("Split: Training",len(training),"Validation",len(validation),"Testing",len(original_test))
# Compact integrity check: the original Stage 1 source, members and
# 1,956/487/463 assignments must be IDENTICAL on every execution.
EXPECTED_SPLIT_SHA = "9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792"
if split_sha != EXPECTED_SPLIT_SHA:
    raise RuntimeError("The rebuilt split differs from frozen Stage 1 image membership.")
print("Stage 1 split integrity verified:", split_sha)

# ----------------------------
# Reproducibility, transforms, and datasets
# ----------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU to perform the controlled performance experiment.")
print("GPU:", torch.cuda.get_device_name(0), "| Visible GPUs:", torch.cuda.device_count())

train_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
test_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)])

class ImageList(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as im:
            image = self.transform(im.convert("RGB"))
        return image, CLASS_TO_ID[row["class"]]

train_ds = ImageList(training, train_tf)
val_ds = ImageList(validation, test_tf)
test_ds = ImageList(original_test, test_tf)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Model structure from Resnet50(2).ipynb: torchvision's pretrained
# ImageNet ResNet-50 backbone, replace the fully connected head with
# the six-class head. All backbone layers remain trainable.
class ResNet50_Model(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            self.model = models.resnet50(weights=weights)
        except Exception as exc:
            if pretrained:
                raise RuntimeError(
                    "Unable to load ImageNet-pretrained ResNet-50 weights. "
                    "Enable Kaggle Internet in Notebook Settings for the first run, "
                    "or ensure torchvision has the weights cached."
                ) from exc
            raise
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)


def make_model(pretrained):
    return ResNet50_Model(num_classes=len(CLASS_NAMES), pretrained=pretrained).to(device)

def evaluate_loss(model, loader, criterion):
    model.eval(); total = 0.0
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            total += criterion(logits, labels.to(device)).item() * len(labels)
    return total / len(loader.dataset)


# Training protocol retains the previous performance notebook settings:
# optimizer Adam lr=1e-4, up to 10 epochs, minimum improvement 1e-4,
# patience 3, best-validation-loss checkpoint, same 350x350 transforms.
# Uses a separate performance output directory, never overwriting
# the original sustainability experiment and its checkpoints.

def prepare_model(seed):
    name = "ResNet-50"
    prefix = f"ResNet50_Torchvision_seed{seed}"
    ckpt_file = OUT / "Model_Checkpoints" / f"{prefix}.pt"
    record_file = OUT / "Model_Checkpoints" / f"{prefix}_training.json"
    if ckpt_file.exists() and record_file.exists():
        saved = json.loads(record_file.read_text())
        if (saved.get("split_sha256") != split_sha or saved.get("seed") != seed
            or saved.get("model") != name or saved.get("architecture") != "torchvision.resnet50"
            or saved.get("image_size") != [32, 32]):
            raise RuntimeError(f"Existing {name} seed {seed} checkpoint belongs to a different experiment")
        model = make_model(pretrained=False)
        obj = torch.load(ckpt_file, map_location="cpu", weights_only=True)
        if (obj.get("split_sha256") != split_sha or obj.get("seed") != seed
                or obj.get("architecture") != "torchvision.resnet50"
                or obj.get("model") != name):
            raise RuntimeError("Checkpoint metadata mismatch")
        model.load_state_dict(obj["state_dict"])
        return model.eval(), saved

    model = make_model(pretrained=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    best_loss, best_epoch, epochs_run, stale = float("inf"), 0, 0, 0
    best_weights = None
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        val_loss = evaluate_loss(model, val_loader, criterion)
        epochs_run = epoch
        print(f"{prefix}: completed epoch {epoch}/{MAX_EPOCHS}; checkpoint selected by validation loss")
        if val_loss < best_loss - MIN_DELTA:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    if best_weights is None:
        raise RuntimeError("No valid best-validation checkpoint was selected")
    model.load_state_dict(best_weights)
    saved = {"model": name, "architecture": "torchvision.resnet50",
             "image_size": [32, 32], "seed": seed, "split_sha256": split_sha,
             "epochs_run": epochs_run, "best_epoch": best_epoch, "best_val_loss": best_loss}
    torch.save({"state_dict": model.state_dict(), "seed": seed, "split_sha256": split_sha,
                "model": name, "architecture": "torchvision.resnet50",
                "best_epoch": best_epoch}, ckpt_file)
    record_file.write_text(json.dumps(saved, indent=2))
    return model.eval(), saved

# One held-out pass per seed: metrics and class probabilities.
PROB_DIR = OUT / "Test_Probabilities"
PROB_DIR.mkdir(parents=True, exist_ok=True)

def evaluate_performance(seed, model):
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model.eval()
    probabilities = []
    true_ids = []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(device))
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
            true_ids.extend(labels.numpy().tolist())
    prob = np.concatenate(probabilities, axis=0)
    true_ids = np.asarray(true_ids, dtype=int)
    pred_ids = np.argmax(prob, axis=1)
    assert prob.shape == (len(original_test), len(CLASS_NAMES))
    assert len(true_ids) == 463
    assert np.allclose(prob.sum(axis=1), 1, atol=1e-5)
    assert np.array_equal(true_ids, np.asarray([CLASS_TO_ID[x['class']] for x in original_test]))
    rows = {"seed": np.repeat(seed, len(original_test)),
            "source_path": [x["source"] for x in original_test],
            "true_class": [CLASS_NAMES[i] for i in true_ids],
            "true_class_id": true_ids,
            "predicted_class": [CLASS_NAMES[i] for i in pred_ids],
            "predicted_class_id": pred_ids,
            "prediction_confidence": prob.max(axis=1)}
    for idx, name in enumerate(CLASS_NAMES):
        rows[f"prob_{name}"] = prob[:, idx]
    df = pd.DataFrame(rows)
    df.to_csv(PROB_DIR / f"ResNet50_seed{seed}_test_probabilities.csv", index=False, float_format="%.10f")
    metrics = {"model": "ResNet50", "seed": seed,
               "accuracy": accuracy_score(true_ids, pred_ids),
               "macro_precision": precision_score(true_ids, pred_ids, labels=list(range(len(CLASS_NAMES))), average="macro", zero_division=0),
               "macro_recall": recall_score(true_ids, pred_ids, labels=list(range(len(CLASS_NAMES))), average="macro", zero_division=0),
               "macro_f1": f1_score(true_ids, pred_ids, labels=list(range(len(CLASS_NAMES))), average="macro", zero_division=0)}
    return metrics, df

results, all_probabilities = [], []
for seed in SEEDS:
    print(f"\n==== ResNet-50 SEED {seed} ====", flush=True)
    set_seed(seed)
    model, train_info = prepare_model(seed)
    row, probs = evaluate_performance(seed, model)
    row.update({"epochs_run": train_info["epochs_run"], "best_epoch": train_info["best_epoch"],
                "best_val_loss": train_info["best_val_loss"]})
    results.append(row)
    all_probabilities.append(probs)
    print(f"ResNet50 seed {seed} | " + " | ".join(f"{key}: {row[key]*100:.2f}%" for key in ("accuracy", "macro_precision", "macro_recall", "macro_f1")))
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
results_df = pd.DataFrame(results)
results_df.to_csv(OUT / "ResNet50_Performance_Per_Seed.csv", index=False)
combined = pd.concat(all_probabilities, ignore_index=True)
combined.to_csv(PROB_DIR / "ResNet50_all_seeds_test_probabilities.csv", index=False, float_format="%.10f")
summary = pd.DataFrame([{
    "model": "ResNet50", "metric": k,
    "mean": float(results_df[k].mean()),
    "sd": float(results_df[k].std(ddof=1)),
    "mean_percent": float(100 * results_df[k].mean()),
    "sd_percent": float(100 * results_df[k].std(ddof=1))
} for k in metric_names])
summary.to_csv(OUT / "ResNet50_5_Seed_Performance_Summary.csv", index=False)
print("\n" + "="*80)
print("FINAL 5-SEED HELD-OUT TEST PERFORMANCE (MEAN +/- SD; percent)")
print("="*80)
print("ResNet50 | " + " | ".join(f"{row.metric}: {row.mean_percent:.2f} +/- {row.sd_percent:.2f}" for row in summary.itertuples()))
print("="*80)

config = {"model": "ResNet50", "seeds": SEEDS, "batch_size": BATCH_SIZE,
          "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "min_delta": MIN_DELTA,
          "optimizer": "Adam", "learning_rate": 1e-4, "image_size": [32, 32],
          "architecture": "torchvision.resnet50 (ImageNet pretrained; all layers fine-tuned)",
          "split_sha256": split_sha,
          "split_counts": {"training": len(train_ds), "validation": len(val_ds), "test": len(test_ds)},
          "macro_average": "unweighted across all six classes; zero_division=0",
          "sd": "sample standard deviation over five seeds (ddof=1)"}
(OUT / "ResNet50_Performance_Configuration.json").write_text(json.dumps(config, indent=2))

report = Document()
report.add_heading("ResNet-50 — Stage 2 Performance Summary", 0)
report.add_paragraph("Held-out 6-class test set (463 images). Five seeds: 7, 49, 112, 345, 476. "
    "Original frozen split, torchvision ResNet50 ImageNet pretrained (no frozen layers), "
    "350x350 transforms, Adam 1e-4, batch size 32, up to 10 epochs, "
    "patience 3, best validation-loss checkpoints. No sustainability measurements.")
table = report.add_table(rows=1, cols=5)
for cell, label in zip(table.rows[0].cells, ["Seed", "Accuracy (%)", "Macro-Precision (%)", "Macro-Recall (%)", "Macro-F1 (%)"]):
    cell.text = label
for row in results_df.itertuples():
    cells = table.add_row().cells
    for cell, value in zip(cells, [str(row.seed)] + [f"{getattr(row,k)*100:.2f}" for k in metric_names]):
        cell.text = value
report.add_heading("Five-seed mean +/- sample SD (%)", level=1)
for row in summary.itertuples():
    report.add_paragraph(f"{row.metric}: {row.mean_percent:.2f} +/- {row.sd_percent:.2f}")
report.add_paragraph("Per-image six-class Softmax probabilities and true/predicted classes are exported separately for each seed and combined across seeds.")
report.save(OUT / "ResNet50_Stage2_Performance_Summary.docx")
print("Results saved under:", OUT)


Source Training: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training 
Source Testing: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
Split: Training 1956 Validation 487 Testing 463
Stage 1 split integrity verified: 9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792
GPU: Tesla T4 | Visible GPUs: 1

==== ResNet-50 SEED 7 ====
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


ResNet50_Torchvision_seed7: completed epoch 1/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed7: completed epoch 2/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed7: completed epoch 3/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed7: completed epoch 4/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed7: completed epoch 5/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed7: completed epoch 6/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed7: completed epoch 7/10; checkpoint selected by validation loss
ResNet50 seed 7 | accuracy: 90.50% | macro_precision: 90.35% | macro_recall: 89.98% | macro_f1: 89.81%

==== ResNet-50 SEED 49 ====
ResNet50_Torchvision_seed49: completed epoch 1/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed49: completed epoch 2/10; checkpoint selected by validation loss
ResNet50_Torchvision_seed49: completed epoch 3/10; checkpoint sel